In [ ]:
pip install statsmodels

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
# Bước 1: Đọc dữ liệu và kiểm tra tính dừng
# Giả sử file 'data.csv' có cột 'date' (là thời gian) và 'value' (giá trị chuỗi thời gian)
data = pd.read_csv('data.csv', parse_dates=['date'], index_col='date')
series = data['value']

# Kiểm tra tính dừng bằng kiểm định ADF
result = adfuller(series)
print('ADF Statistic: %f' % result[0])
print('p-value: %f' % result[1])

# Nếu p-value > 0.05, chuỗi không dừng, tiến hành lấy sai phân
if result[1] > 0.05:
    series_diff = series.diff().dropna()
    d = 1  # Thường là 1 nếu chỉ cần lấy sai phân 1 lần
else:
    series_diff = series
    d = 0

# Vẽ đồ thị để xem lại chuỗi đã lấy sai phân
plt.figure(figsize=(10, 4))
plt.plot(series_diff)
plt.title('Chuỗi sau khi lấy sai phân')
plt.show()

# Bước 2: Xác định tham số p và q bằng đồ thị ACF và PACF
fig, ax = plt.subplots(1,2, figsize=(16,4))
plot_acf(series_diff, ax=ax[0], lags=5)
plot_pacf(series_diff, ax=ax[1], lags=3)
plt.show()

# Dựa vào đồ thị, giả sử ta chọn p=1 và q=1 (trong thực tế cần cân nhắc kỹ lưỡng)
p, q = 1, 1

# Bước 3: Xây dựng và huấn luyện mô hình ARIMA
model = ARIMA(series, order=(p, d, q))
model_fit = model.fit()

# Hiển thị tóm tắt kết quả mô hình
print(model_fit.summary())

# Bước 4: Dự báo các giá trị tương lai
forecast_steps = 10  # Số bước dự báo
forecast = model_fit.forecast(steps=forecast_steps)
print("Dự báo {} giá trị tiếp theo:".format(forecast_steps))
print(forecast)

# Vẽ đồ thị dự báo kèm theo dữ liệu gốc
plt.figure(figsize=(10, 4))
plt.plot(series, label='Dữ liệu gốc')
plt.plot(forecast, label='Dự báo', color='red')
plt.title('Dự báo với mô hình ARIMA')
plt.legend()
plt.show()